<a href="https://colab.research.google.com/github/toussaintMushamuka/afyaAssist/blob/main/dense_forest_oracle.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# ============================================================
# FULL-GRID SPATIOTEMPORAL FORECASTING MODEL
#
# Data:
#     108 months
#     H x W regular 1-km grid
#     64-dimensional embedding at every grid point
#
# Training:
#     48 historical months -> 12 future months
#
# Final forecast:
#     5 x 12 months = 60 future months
#
# No mask
# No spatial crops
# Entire rectangular grid is processed simultaneously
# ============================================================

In [ ]:
import math
import copy
import random
from dataclasses import dataclass

import numpy as np

import torch
import torch.nn as nn
import torch.nn.functional as F

from torch.utils.data import Dataset, DataLoader

In [ ]:

# ============================================================
# 1. CONFIGURATION
# ============================================================

@dataclass
class Config:

    # --------------------------------------------------------
    # DATA
    # --------------------------------------------------------

    total_months: int = 108

    embedding_dim: int = 64

    # --------------------------------------------------------
    # FORECASTING
    # --------------------------------------------------------

    # Number of historical months used by the model
    input_length: int = 48

    # Number of months directly predicted by one model call
    chunk_horizon: int = 12

    # Final required forecast
    final_horizon: int = 60

    # --------------------------------------------------------
    # SPATIAL CNN
    # --------------------------------------------------------

    # Dimension of compressed spatial representation
    d_model: int = 96

    # TOTAL number of convolution operations in
    # the spatial encoder.
    #
    # Decoder uses the same depth.
    #
    # Minimum = 3
    #
    # Good values to test:
    #
    # 3, 5, 7, 9
    #
    num_spatial_conv_layers: int = 7

    # --------------------------------------------------------
    # TEMPORAL TRANSFORMER
    # --------------------------------------------------------

    nhead: int = 4

    num_transformer_encoder_layers: int = 2

    num_transformer_decoder_layers: int = 1

    transformer_feedforward_dim: int = 192

    dropout: float = 0.10

    # --------------------------------------------------------
    # TRAINING
    # --------------------------------------------------------

    # Full national grid can be large.
    # Start with 1.
    batch_size: int = 1

    epochs: int = 100

    learning_rate: float = 1e-4

    weight_decay: float = 1e-4

    gradient_clip: float = 1.0

    # --------------------------------------------------------
    # EMBEDDING LOSS
    # --------------------------------------------------------

    mse_weight: float = 0.5

    cosine_weight: float = 0.5

    # Set True if cosine geometry is meaningful for the
    # original embeddings.
    normalize_embeddings: bool = True

    # --------------------------------------------------------
    # REPRODUCIBILITY
    # --------------------------------------------------------

    seed: int = 42


cfg = Config()


# ============================================================
# 2. REPRODUCIBILITY
# ============================================================

def set_seed(seed):

    random.seed(seed)

    np.random.seed(seed)

    torch.manual_seed(seed)

    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


set_seed(cfg.seed)


# ============================================================
# 3. DEVICE
# ============================================================

device = torch.device(
    "cuda"
    if torch.cuda.is_available()
    else "cpu"
)

print("Device:", device)


# ============================================================
# 4. OPTIONAL CONVERSION BETWEEN POINTS AND GRID
# ============================================================
#
# If your original data have shape:
#
#     [N, 108, 64]
#
# and the points are stored in row-major grid order,
# with:
#
#     N = H * W
#
# this converts them into:
#
#     [108, 64, H, W]
#
# ============================================================

def points_to_grid(
    point_data,
    height,
    width,
):
    """
    Parameters
    ----------
    point_data:
        Tensor [N, T, C]

    height:
        Number of grid rows.

    width:
        Number of grid columns.

    Returns
    -------
    grid:
        Tensor [T, C, H, W]

    IMPORTANT
    ---------
    Assumes points are ordered row-by-row.
    """

    N, T, C = point_data.shape

    if N != height * width:

        raise ValueError(
            f"N={N}, but H*W={height*width}"
        )

    grid = point_data.reshape(
        height,
        width,
        T,
        C,
    )

    grid = grid.permute(
        2,  # time
        3,  # embedding dimension
        0,  # row
        1,  # column
    ).contiguous()

    return grid


def grid_to_points(grid):
    """
    Convert:

        [T, C, H, W]

    to:

        [N, T, C]

    where:

        N = H * W
    """

    T, C, H, W = grid.shape

    points = grid.permute(
        2,
        3,
        0,
        1,
    ).contiguous()

    points = points.reshape(
        H * W,
        T,
        C,
    )

    return points


# ============================================================
# 5. LOAD DATA
# ============================================================
#
# Expected final shape:
#
#     data.shape = [108, 64, H, W]
#
# Example:
#
# data = torch.load(
#     "embeddings_grid.pt",
#     map_location="cpu"
# ).float()
#
# ============================================================


# ============================================================
# 6. NORMALIZE EMBEDDINGS
# ============================================================

def normalize_embeddings(data):
    """
    Normalize each 64-dimensional embedding independently.

    Input:
        [T, 64, H, W]

    Output:
        [T, 64, H, W]

    Normalization is along dimension 1,
    which is the embedding dimension.
    """

    return F.normalize(
        data,
        p=2,
        dim=1,
        eps=1e-8,
    )


# After loading:
#
# if cfg.normalize_embeddings:
#     data = normalize_embeddings(data)


# ============================================================
# 7. TEMPORAL WINDOWS
# ============================================================
#
# Model:
#
#     48 months -> 12 months
#
# With 108 months:
#
#     108 - 48 - 12 + 1
#     = 49 possible temporal windows
#
# ============================================================


# Training uses information up to month 84.
#
# Example:
#
#   1:48  -> 49:60
#   ...
#   25:72 -> 73:84

train_starts = list(range(0, 25))


# Validation:
#
#   37:84 -> 85:96

val_starts = [36]


# Test:
#
#   49:96 -> 97:108

test_starts = [48]


# After model selection, final production training
# can use all 49 available windows.

all_starts = list(range(49))


# ============================================================
# 8. DATASET
# ============================================================

class FullGridTemporalDataset(Dataset):

    def __init__(
        self,
        data,
        time_starts,
        input_length=48,
        horizon=12,
    ):
        """
        Parameters
        ----------
        data:
            [T, 64, H, W]

        time_starts:
            List of starting time indices.

        input_length:
            Historical months.

        horizon:
            Direct forecast horizon.
        """

        self.data = data

        self.time_starts = time_starts

        self.input_length = input_length

        self.horizon = horizon

    def __len__(self):

        return len(
            self.time_starts
        )

    def __getitem__(self, index):

        start = self.time_starts[index]

        input_end = (
            start
            +
            self.input_length
        )

        target_end = (
            input_end
            +
            self.horizon
        )

        # Historical country grids
        #
        # [48, 64, H, W]

        X = self.data[
            start:input_end
        ]

        # Future country grids
        #
        # [12, 64, H, W]

        Y = self.data[
            input_end:target_end
        ]

        return X, Y


# ============================================================
# 9. GROUP-NORMALIZATION HELPER
# ============================================================

def get_num_groups(
    channels,
    max_groups=8,
):
    """
    Find a valid GroupNorm group count.
    """

    for g in range(
        min(max_groups, channels),
        0,
        -1,
    ):

        if channels % g == 0:

            return g

    return 1


# ============================================================
# 10. DISTRIBUTE EXTRA CNN LAYERS
# ============================================================

def distribute_layers(
    number_of_layers,
    number_of_stages=3,
):
    """
    Distribute additional convolution layers
    across the three spatial scales.

    Example
    -------
    number_of_layers = 4

    returns:

        [2, 1, 1]
    """

    base = (
        number_of_layers
        //
        number_of_stages
    )

    remainder = (
        number_of_layers
        %
        number_of_stages
    )

    result = []

    for stage in range(
        number_of_stages
    ):

        n = base

        if stage < remainder:
            n += 1

        result.append(n)

    return result


# ============================================================
# 11. ONE RESIDUAL CNN LAYER
# ============================================================

class ResidualConvLayer(nn.Module):
    """
    One spatial 3x3 convolution.

    Shape is preserved:

        [B, C, H, W]
             ->
        [B, C, H, W]
    """

    def __init__(
        self,
        channels,
        dropout=0.1,
    ):

        super().__init__()

        self.conv = nn.Conv2d(
            channels,
            channels,
            kernel_size=3,
            stride=1,
            padding=1,
        )

        self.norm = nn.GroupNorm(
            get_num_groups(
                channels
            ),
            channels,
        )

        self.dropout = nn.Dropout2d(
            dropout
        )

    def forward(self, x):

        residual = x

        x = self.conv(x)

        x = self.norm(x)

        x = F.gelu(x)

        x = self.dropout(x)

        # Residual connection improves
        # optimization for deeper CNNs.

        return x + residual


# ============================================================
# 12. SPATIAL ENCODER
# ============================================================
#
# The encoder transforms:
#
#     [64, H, W]
#
# into:
#
#     [d_model, H/4, W/4]
#
#
# It contains:
#
#     1 input convolution
#     2 stride-2 convolutions
#
# plus optional residual convolutions.
#
# Therefore:
#
#     num_conv_layers >= 3
#
# ============================================================

class SpatialEncoder(nn.Module):

    def __init__(
        self,
        embedding_dim=64,
        d_model=96,
        num_conv_layers=7,
        dropout=0.1,
    ):

        super().__init__()

        if num_conv_layers < 3:

            raise ValueError(
                "num_conv_layers must be >= 3"
            )

        self.num_conv_layers = (
            num_conv_layers
        )

        # ----------------------------------------------------
        # Three convolutions are structurally required:
        #
        # 1 input convolution
        # 2 downsampling convolutions
        # ----------------------------------------------------

        extra_layers = (
            num_conv_layers - 3
        )

        stage_counts = distribute_layers(
            extra_layers,
            number_of_stages=3,
        )

        stage0_count = stage_counts[0]

        stage1_count = stage_counts[1]

        stage2_count = stage_counts[2]

        # ----------------------------------------------------
        # ORIGINAL RESOLUTION
        #
        # [64, H, W]
        #        ->
        # [64, H, W]
        # ----------------------------------------------------

        self.input_conv = nn.Sequential(

            nn.Conv2d(
                embedding_dim,
                64,
                kernel_size=3,
                stride=1,
                padding=1,
            ),

            nn.GroupNorm(
                get_num_groups(64),
                64,
            ),

            nn.GELU(),
        )

        self.stage0 = nn.Sequential(
            *[
                ResidualConvLayer(
                    channels=64,
                    dropout=dropout,
                )
                for _ in range(
                    stage0_count
                )
            ]
        )

        # ----------------------------------------------------
        # FIRST DOWNSAMPLING
        #
        # H x W
        #
        # ->
        #
        # H/2 x W/2
        # ----------------------------------------------------

        self.downsample1 = nn.Sequential(

            nn.Conv2d(
                64,
                80,
                kernel_size=4,
                stride=2,
                padding=1,
            ),

            nn.GroupNorm(
                get_num_groups(80),
                80,
            ),

            nn.GELU(),
        )

        self.stage1 = nn.Sequential(
            *[
                ResidualConvLayer(
                    channels=80,
                    dropout=dropout,
                )
                for _ in range(
                    stage1_count
                )
            ]
        )

        # ----------------------------------------------------
        # SECOND DOWNSAMPLING
        #
        # H/2 x W/2
        #
        # ->
        #
        # H/4 x W/4
        # ----------------------------------------------------

        self.downsample2 = nn.Sequential(

            nn.Conv2d(
                80,
                d_model,
                kernel_size=4,
                stride=2,
                padding=1,
            ),

            nn.GroupNorm(
                get_num_groups(
                    d_model
                ),
                d_model,
            ),

            nn.GELU(),
        )

        self.stage2 = nn.Sequential(
            *[
                ResidualConvLayer(
                    channels=d_model,
                    dropout=dropout,
                )
                for _ in range(
                    stage2_count
                )
            ]
        )

    def forward(self, x):

        # Full 1-km resolution

        x = self.input_conv(x)

        x = self.stage0(x)

        # Half resolution

        x = self.downsample1(x)

        x = self.stage1(x)

        # Quarter resolution

        x = self.downsample2(x)

        x = self.stage2(x)

        return x


# ============================================================
# 13. SPATIAL DECODER
# ============================================================
#
# Performs the reverse transformation:
#
#     [d_model, H/4, W/4]
#
# ->
#
#     [64, H, W]
#
# ============================================================

class SpatialDecoder(nn.Module):

    def __init__(
        self,
        d_model=96,
        embedding_dim=64,
        num_conv_layers=7,
        dropout=0.1,
    ):

        super().__init__()

        if num_conv_layers < 3:

            raise ValueError(
                "num_conv_layers must be >= 3"
            )

        self.num_conv_layers = (
            num_conv_layers
        )

        extra_layers = (
            num_conv_layers - 3
        )

        stage_counts = distribute_layers(
            extra_layers,
            number_of_stages=3,
        )

        stage0_count = stage_counts[0]

        stage1_count = stage_counts[1]

        stage2_count = stage_counts[2]

        # ----------------------------------------------------
        # LATENT SPATIAL RESOLUTION
        # ----------------------------------------------------

        self.stage0 = nn.Sequential(
            *[
                ResidualConvLayer(
                    channels=d_model,
                    dropout=dropout,
                )
                for _ in range(
                    stage0_count
                )
            ]
        )

        # ----------------------------------------------------
        # H/4 -> H/2
        # ----------------------------------------------------

        self.upsample1 = nn.Sequential(

            nn.ConvTranspose2d(
                d_model,
                80,
                kernel_size=4,
                stride=2,
                padding=1,
            ),

            nn.GroupNorm(
                get_num_groups(80),
                80,
            ),

            nn.GELU(),
        )

        self.stage1 = nn.Sequential(
            *[
                ResidualConvLayer(
                    channels=80,
                    dropout=dropout,
                )
                for _ in range(
                    stage1_count
                )
            ]
        )

        # ----------------------------------------------------
        # H/2 -> H
        # ----------------------------------------------------

        self.upsample2 = nn.Sequential(

            nn.ConvTranspose2d(
                80,
                64,
                kernel_size=4,
                stride=2,
                padding=1,
            ),

            nn.GroupNorm(
                get_num_groups(64),
                64,
            ),

            nn.GELU(),
        )

        self.stage2 = nn.Sequential(
            *[
                ResidualConvLayer(
                    channels=64,
                    dropout=dropout,
                )
                for _ in range(
                    stage2_count
                )
            ]
        )

        # ----------------------------------------------------
        # Restore original 64-dimensional embedding
        # ----------------------------------------------------

        self.output_conv = nn.Conv2d(
            64,
            embedding_dim,
            kernel_size=3,
            stride=1,
            padding=1,
        )

    def forward(self, x):

        x = self.stage0(x)

        x = self.upsample1(x)

        x = self.stage1(x)

        x = self.upsample2(x)

        x = self.stage2(x)

        x = self.output_conv(x)

        return x


# ============================================================
# 14. TEMPORAL POSITIONAL ENCODING
# ============================================================

class SinusoidalPositionalEncoding(
    nn.Module
):

    def __init__(
        self,
        d_model,
        max_len=256,
    ):

        super().__init__()

        position = torch.arange(
            max_len,
            dtype=torch.float32,
        ).unsqueeze(1)

        div_term = torch.exp(

            torch.arange(
                0,
                d_model,
                2,
                dtype=torch.float32,
            )

            *

            (
                -math.log(10000.0)
                /
                d_model
            )
        )

        pe = torch.zeros(
            max_len,
            d_model,
        )

        pe[:, 0::2] = torch.sin(
            position
            *
            div_term
        )

        pe[:, 1::2] = torch.cos(
            position
            *
            div_term
        )

        self.register_buffer(
            "pe",
            pe.unsqueeze(0),
        )

    def forward(self, x):

        return (
            x
            +
            self.pe[
                :,
                :x.shape[1]
            ]
        )


# ============================================================
# 15. SPATIAL PADDING
# ============================================================
#
# Two stride-2 operations require H and W to be
# divisible by four.
#
# If they are not, padding is applied internally.
# ============================================================

def pad_to_multiple(
    x,
    multiple=4,
):

    H = x.shape[-2]

    W = x.shape[-1]

    pad_h = (
        multiple
        -
        H % multiple
    ) % multiple

    pad_w = (
        multiple
        -
        W % multiple
    ) % multiple

    x = F.pad(
        x,
        (
            0,
            pad_w,
            0,
            pad_h,
        ),
    )

    return (
        x,
        H,
        W,
    )


# ============================================================
# 16. COMPLETE MODEL
# ============================================================

class FullGridSpatialTemporalTransformer(
    nn.Module
):

    def __init__(
        self,
        embedding_dim=64,
        d_model=96,
        num_spatial_conv_layers=7,
        nhead=4,
        num_transformer_encoder_layers=2,
        num_transformer_decoder_layers=1,
        transformer_feedforward_dim=192,
        dropout=0.1,
        input_length=48,
        horizon=12,
        normalize_output=True,
    ):

        super().__init__()

        if d_model % nhead != 0:

            raise ValueError(
                "d_model must be divisible by nhead"
            )

        self.embedding_dim = (
            embedding_dim
        )

        self.d_model = (
            d_model
        )

        self.input_length = (
            input_length
        )

        self.horizon = (
            horizon
        )

        self.normalize_output = (
            normalize_output
        )

        self.num_spatial_conv_layers = (
            num_spatial_conv_layers
        )

        # ====================================================
        # SPATIAL ENCODER
        # ====================================================

        self.spatial_encoder = (
            SpatialEncoder(
                embedding_dim=embedding_dim,
                d_model=d_model,
                num_conv_layers=(
                    num_spatial_conv_layers
                ),
                dropout=dropout,
            )
        )

        # ====================================================
        # SPATIAL DECODER
        # ====================================================

        self.spatial_decoder = (
            SpatialDecoder(
                d_model=d_model,
                embedding_dim=embedding_dim,
                num_conv_layers=(
                    num_spatial_conv_layers
                ),
                dropout=dropout,
            )
        )

        # ====================================================
        # TEMPORAL POSITION
        # ====================================================

        self.temporal_position = (
            SinusoidalPositionalEncoding(
                d_model=d_model,
                max_len=256,
            )
        )

        # ====================================================
        # TEMPORAL TRANSFORMER ENCODER
        # ====================================================

        encoder_layer = (
            nn.TransformerEncoderLayer(

                d_model=d_model,

                nhead=nhead,

                dim_feedforward=(
                    transformer_feedforward_dim
                ),

                dropout=dropout,

                activation="gelu",

                batch_first=True,

                norm_first=True,
            )
        )

        self.temporal_encoder = (
            nn.TransformerEncoder(

                encoder_layer,

                num_layers=(
                    num_transformer_encoder_layers
                ),
            )
        )

        # ====================================================
        # LEARNED FUTURE MONTH QUERIES
        # ====================================================
        #
        # One trainable vector for each directly
        # predicted future month.
        # ====================================================

        self.future_queries = (
            nn.Parameter(
                torch.randn(
                    horizon,
                    d_model,
                )
                *
                0.02
            )
        )

        # ====================================================
        # TEMPORAL TRANSFORMER DECODER
        # ====================================================

        decoder_layer = (
            nn.TransformerDecoderLayer(

                d_model=d_model,

                nhead=nhead,

                dim_feedforward=(
                    transformer_feedforward_dim
                ),

                dropout=dropout,

                activation="gelu",

                batch_first=True,

                norm_first=True,
            )
        )

        self.temporal_decoder = (
            nn.TransformerDecoder(

                decoder_layer,

                num_layers=(
                    num_transformer_decoder_layers
                ),
            )
        )

    def forward(self, x):
        """
        Parameters
        ----------
        x:
            [B, 48, 64, H, W]

        Returns
        -------
        prediction:
            [B, 12, 64, H, W]
        """

        B, L, C, H, W = x.shape

        if L != self.input_length:

            raise ValueError(
                f"Expected {self.input_length} months, "
                f"received {L}"
            )

        if C != self.embedding_dim:

            raise ValueError(
                f"Expected embedding dimension "
                f"{self.embedding_dim}, received {C}"
            )

        # ====================================================
        # STEP 1
        #
        # Treat every month as one spatial image.
        #
        # [B, L, 64, H, W]
        #
        # ->
        #
        # [B*L, 64, H, W]
        # ====================================================

        spatial_input = x.reshape(
            B * L,
            C,
            H,
            W,
        )

        # ====================================================
        # STEP 2
        #
        # Make H and W divisible by 4.
        # ====================================================

        (
            spatial_input,
            original_H,
            original_W,
        ) = pad_to_multiple(
            spatial_input,
            multiple=4,
        )

        Hp = spatial_input.shape[-2]

        Wp = spatial_input.shape[-1]

        # ====================================================
        # STEP 3
        #
        # CNN spatial encoding.
        #
        # [B*L, 64, Hp, Wp]
        #
        # ->
        #
        # [B*L, d_model, Hp/4, Wp/4]
        # ====================================================

        latent = self.spatial_encoder(
            spatial_input
        )

        _, D, Hc, Wc = (
            latent.shape
        )

        # ====================================================
        # STEP 4
        #
        # Restore explicit time dimension.
        #
        # [B*L,D,Hc,Wc]
        #
        # ->
        #
        # [B,L,D,Hc,Wc]
        # ====================================================

        latent = latent.reshape(
            B,
            L,
            D,
            Hc,
            Wc,
        )

        # ====================================================
        # STEP 5
        #
        # For every compressed spatial position,
        # build a 48-month temporal sequence.
        #
        # [B,L,D,Hc,Wc]
        #
        # ->
        #
        # [B,Hc,Wc,L,D]
        #
        # ->
        #
        # [B*Hc*Wc,L,D]
        # ====================================================

        temporal = latent.permute(
            0,
            3,
            4,
            1,
            2,
        ).contiguous()

        temporal = temporal.reshape(
            B * Hc * Wc,
            L,
            D,
        )

        # ====================================================
        # STEP 6
        #
        # Add temporal position information.
        # ====================================================

        temporal = (
            self.temporal_position(
                temporal
            )
        )

        # ====================================================
        # STEP 7
        #
        # Encode historical temporal dynamics.
        #
        # [B*Hc*Wc,48,D]
        #
        # ->
        #
        # [B*Hc*Wc,48,D]
        # ====================================================

        memory = self.temporal_encoder(
            temporal
        )

        # ====================================================
        # STEP 8
        #
        # Create 12 future queries for each spatial
        # latent position.
        #
        # [12,D]
        #
        # ->
        #
        # [B*Hc*Wc,12,D]
        # ====================================================

        future_queries = (
            self.future_queries
            .unsqueeze(0)
            .expand(
                B * Hc * Wc,
                -1,
                -1,
            )
        )

        # ====================================================
        # STEP 9
        #
        # Predict future temporal latent states.
        #
        # [B*Hc*Wc,12,D]
        # ====================================================

        future_latent = (
            self.temporal_decoder(

                tgt=future_queries,

                memory=memory,
            )
        )

        # ====================================================
        # STEP 10
        #
        # Restore spatial organization.
        #
        # ->
        #
        # [B,12,D,Hc,Wc]
        # ====================================================

        future_latent = (
            future_latent.reshape(
                B,
                Hc,
                Wc,
                self.horizon,
                D,
            )
        )

        future_latent = (
            future_latent.permute(
                0,
                3,
                4,
                1,
                2,
            )
            .contiguous()
        )

        # ====================================================
        # STEP 11
        #
        # Decode each future month spatially.
        #
        # [B,12,D,Hc,Wc]
        #
        # ->
        #
        # [B*12,D,Hc,Wc]
        # ====================================================

        decoder_input = (
            future_latent.reshape(
                B * self.horizon,
                D,
                Hc,
                Wc,
            )
        )

        prediction = (
            self.spatial_decoder(
                decoder_input
            )
        )

        # ====================================================
        # STEP 12
        #
        # Restore batch/time dimensions.
        #
        # ->
        #
        # [B,12,64,Hp,Wp]
        # ====================================================

        prediction = prediction.reshape(
            B,
            self.horizon,
            self.embedding_dim,
            Hp,
            Wp,
        )

        # ====================================================
        # STEP 13
        #
        # Remove any padding added earlier.
        # ====================================================

        prediction = prediction[
            :,
            :,
            :,
            :original_H,
            :original_W,
        ]

        # ====================================================
        # STEP 14
        #
        # Put predicted embeddings back on unit sphere
        # when cosine geometry is used.
        # ====================================================

        if self.normalize_output:

            prediction = F.normalize(
                prediction,
                p=2,
                dim=2,
                eps=1e-8,
            )

        return prediction


# ============================================================
# 17. CREATE MODEL
# ============================================================

def create_model(cfg):

    return FullGridSpatialTemporalTransformer(

        embedding_dim=(
            cfg.embedding_dim
        ),

        d_model=(
            cfg.d_model
        ),

        num_spatial_conv_layers=(
            cfg.num_spatial_conv_layers
        ),

        nhead=(
            cfg.nhead
        ),

        num_transformer_encoder_layers=(
            cfg.num_transformer_encoder_layers
        ),

        num_transformer_decoder_layers=(
            cfg.num_transformer_decoder_layers
        ),

        transformer_feedforward_dim=(
            cfg.transformer_feedforward_dim
        ),

        dropout=(
            cfg.dropout
        ),

        input_length=(
            cfg.input_length
        ),

        horizon=(
            cfg.chunk_horizon
        ),

        normalize_output=(
            cfg.normalize_embeddings
        ),
    )


model = create_model(cfg).to(device)


# ============================================================
# 18. COUNT PARAMETERS
# ============================================================

def count_parameters(model):

    return sum(
        p.numel()
        for p in model.parameters()
        if p.requires_grad
    )


print(
    "Trainable parameters:",
    f"{count_parameters(model):,}"
)

print(
    "Spatial CNN depth:",
    cfg.num_spatial_conv_layers
)


# ============================================================
# 19. LOSS
# ============================================================

class EmbeddingForecastLoss(nn.Module):

    def __init__(
        self,
        mse_weight=0.5,
        cosine_weight=0.5,
    ):

        super().__init__()

        self.mse_weight = (
            mse_weight
        )

        self.cosine_weight = (
            cosine_weight
        )

    def forward(
        self,
        prediction,
        target,
    ):
        """
        prediction:
            [B,12,64,H,W]

        target:
            [B,12,64,H,W]
        """

        # ----------------------------------------------------
        # Euclidean component
        # ----------------------------------------------------

        mse_loss = F.mse_loss(
            prediction,
            target,
        )

        # ----------------------------------------------------
        # Cosine component
        #
        # dim=2 is the 64-dimensional embedding axis.
        # ----------------------------------------------------

        cosine_similarity = (
            F.cosine_similarity(
                prediction,
                target,
                dim=2,
                eps=1e-8,
            )
        )

        cosine_loss = (
            1.0
            -
            cosine_similarity
        ).mean()

        # ----------------------------------------------------
        # Combined objective
        # ----------------------------------------------------

        total_loss = (

            self.mse_weight
            *
            mse_loss

            +

            self.cosine_weight
            *
            cosine_loss
        )

        return {

            "loss":
                total_loss,

            "mse":
                mse_loss,

            "cosine_loss":
                cosine_loss,

            "cosine_similarity":
                cosine_similarity.mean(),
        }


criterion = EmbeddingForecastLoss(

    mse_weight=(
        cfg.mse_weight
    ),

    cosine_weight=(
        cfg.cosine_weight
    ),
)


# ============================================================
# 20. DATASETS AND LOADERS
# ============================================================
#
# Run this after `data` has been loaded.
# ============================================================

def make_loaders(
    data,
    cfg,
):

    train_dataset = (
        FullGridTemporalDataset(

            data=data,

            time_starts=train_starts,

            input_length=(
                cfg.input_length
            ),

            horizon=(
                cfg.chunk_horizon
            ),
        )
    )

    val_dataset = (
        FullGridTemporalDataset(

            data=data,

            time_starts=val_starts,

            input_length=(
                cfg.input_length
            ),

            horizon=(
                cfg.chunk_horizon
            ),
        )
    )

    test_dataset = (
        FullGridTemporalDataset(

            data=data,

            time_starts=test_starts,

            input_length=(
                cfg.input_length
            ),

            horizon=(
                cfg.chunk_horizon
            ),
        )
    )

    pin_memory = (
        device.type == "cuda"
    )

    train_loader = DataLoader(

        train_dataset,

        batch_size=(
            cfg.batch_size
        ),

        shuffle=True,

        num_workers=0,

        pin_memory=pin_memory,
    )

    val_loader = DataLoader(

        val_dataset,

        batch_size=1,

        shuffle=False,

        num_workers=0,

        pin_memory=pin_memory,
    )

    test_loader = DataLoader(

        test_dataset,

        batch_size=1,

        shuffle=False,

        num_workers=0,

        pin_memory=pin_memory,
    )

    return (
        train_loader,
        val_loader,
        test_loader,
    )


# ============================================================
# 21. OPTIMIZER
# ============================================================

optimizer = torch.optim.AdamW(

    model.parameters(),

    lr=(
        cfg.learning_rate
    ),

    weight_decay=(
        cfg.weight_decay
    ),
)


scheduler = (
    torch.optim.lr_scheduler
    .CosineAnnealingLR(

        optimizer,

        T_max=(
            cfg.epochs
        ),

        eta_min=1e-6,
    )
)


# ============================================================
# 22. MIXED PRECISION
# ============================================================

use_amp = (
    device.type == "cuda"
)

amp_dtype = (
    torch.bfloat16
    if (
        use_amp
        and
        torch.cuda.is_bf16_supported()
    )
    else
    torch.float16
)


scaler = torch.amp.GradScaler(
    "cuda",
    enabled=(
        use_amp
        and
        amp_dtype == torch.float16
    ),
)


# ============================================================
# 23. TRAIN ONE EPOCH
# ============================================================

def train_one_epoch(
    model,
    loader,
    optimizer,
    criterion,
    scaler,
    device,
    gradient_clip,
):

    model.train()

    total_loss = 0.0

    total_mse = 0.0

    total_cosine = 0.0

    number_batches = 0

    for X, Y in loader:

        X = X.to(
            device,
            non_blocking=True,
        )

        Y = Y.to(
            device,
            non_blocking=True,
        )

        optimizer.zero_grad(
            set_to_none=True
        )

        # ----------------------------------------------------
        # Forward
        # ----------------------------------------------------

        with torch.autocast(
            device_type=device.type,
            dtype=amp_dtype,
            enabled=use_amp,
        ):

            prediction = model(X)

            metrics = criterion(
                prediction,
                Y,
            )

            loss = metrics["loss"]

        # ----------------------------------------------------
        # Backpropagation
        # ----------------------------------------------------

        if scaler.is_enabled():

            scaler.scale(
                loss
            ).backward()

            scaler.unscale_(
                optimizer
            )

            torch.nn.utils.clip_grad_norm_(
                model.parameters(),
                max_norm=gradient_clip,
            )

            scaler.step(
                optimizer
            )

            scaler.update()

        else:

            loss.backward()

            torch.nn.utils.clip_grad_norm_(
                model.parameters(),
                max_norm=gradient_clip,
            )

            optimizer.step()

        total_loss += (
            loss.item()
        )

        total_mse += (
            metrics["mse"].item()
        )

        total_cosine += (
            metrics[
                "cosine_similarity"
            ].item()
        )

        number_batches += 1

    return {

        "loss":
            total_loss
            /
            number_batches,

        "mse":
            total_mse
            /
            number_batches,

        "cosine_similarity":
            total_cosine
            /
            number_batches,
    }


# ============================================================
# 24. VALIDATION / TEST
# ============================================================

@torch.no_grad()
def evaluate(
    model,
    loader,
    criterion,
    device,
):

    model.eval()

    total_loss = 0.0

    total_mse = 0.0

    total_cosine = 0.0

    number_batches = 0

    for X, Y in loader:

        X = X.to(device)

        Y = Y.to(device)

        with torch.autocast(
            device_type=device.type,
            dtype=amp_dtype,
            enabled=use_amp,
        ):

            prediction = model(X)

            metrics = criterion(
                prediction,
                Y,
            )

        total_loss += (
            metrics["loss"].item()
        )

        total_mse += (
            metrics["mse"].item()
        )

        total_cosine += (
            metrics[
                "cosine_similarity"
            ].item()
        )

        number_batches += 1

    return {

        "loss":
            total_loss
            /
            number_batches,

        "mse":
            total_mse
            /
            number_batches,

        "cosine_similarity":
            total_cosine
            /
            number_batches,
    }


# ============================================================
# 25. TRAINING LOOP
# ============================================================

def fit_model(
    model,
    train_loader,
    val_loader,
    cfg,
    checkpoint_path="best_model.pt",
):

    criterion = EmbeddingForecastLoss(
        mse_weight=(
            cfg.mse_weight
        ),
        cosine_weight=(
            cfg.cosine_weight
        ),
    )

    optimizer = torch.optim.AdamW(

        model.parameters(),

        lr=(
            cfg.learning_rate
        ),

        weight_decay=(
            cfg.weight_decay
        ),
    )

    scheduler = (
        torch.optim.lr_scheduler
        .CosineAnnealingLR(

            optimizer,

            T_max=(
                cfg.epochs
            ),

            eta_min=1e-6,
        )
    )

    use_amp_local = (
        device.type == "cuda"
    )

    scaler_local = torch.amp.GradScaler(
        "cuda",
        enabled=(
            use_amp_local
            and
            amp_dtype
            ==
            torch.float16
        ),
    )

    best_val_loss = float(
        "inf"
    )

    best_epoch = None

    for epoch in range(
        1,
        cfg.epochs + 1,
    ):

        train_metrics = (
            train_one_epoch(

                model=model,

                loader=train_loader,

                optimizer=optimizer,

                criterion=criterion,

                scaler=scaler_local,

                device=device,

                gradient_clip=(
                    cfg.gradient_clip
                ),
            )
        )

        val_metrics = evaluate(

            model=model,

            loader=val_loader,

            criterion=criterion,

            device=device,
        )

        scheduler.step()

        print(
            f"Epoch {epoch:03d} | "
            f"Train loss="
            f"{train_metrics['loss']:.6f} | "
            f"Train cos="
            f"{train_metrics['cosine_similarity']:.4f} | "
            f"Val loss="
            f"{val_metrics['loss']:.6f} | "
            f"Val cos="
            f"{val_metrics['cosine_similarity']:.4f}"
        )

        if (
            val_metrics["loss"]
            <
            best_val_loss
        ):

            best_val_loss = (
                val_metrics["loss"]
            )

            best_epoch = epoch

            torch.save(
                {
                    "epoch":
                        epoch,

                    "model_state_dict":
                        model.state_dict(),

                    "optimizer_state_dict":
                        optimizer.state_dict(),

                    "validation_loss":
                        best_val_loss,

                    "config":
                        vars(cfg),
                },

                checkpoint_path,
            )

    return (
        best_epoch,
        best_val_loss,
    )


# ============================================================
# 26. COSINE SIMILARITY BY FORECAST MONTH
# ============================================================

@torch.no_grad()
def cosine_by_horizon(
    model,
    loader,
    device,
):

    model.eval()

    scores = []

    for X, Y in loader:

        X = X.to(device)

        Y = Y.to(device)

        prediction = model(X)

        # ----------------------------------------------------
        # Cosine similarity across 64-dimensional embeddings
        #
        # output:
        #
        # [B,12,H,W]
        # ----------------------------------------------------

        similarity = F.cosine_similarity(
            prediction,
            Y,
            dim=2,
            eps=1e-8,
        )

        # Average:
        #
        # batch
        # H
        # W
        #
        # while preserving horizon.
        #

        similarity = similarity.mean(
            dim=(0, 2, 3)
        )

        scores.append(
            similarity.cpu()
        )

    return torch.stack(
        scores
    ).mean(
        dim=0
    )


# ============================================================
# 27. ONE 12-MONTH FORECAST
# ============================================================

@torch.no_grad()
def forecast_next_12_months(
    model,
    history,
    device,
):
    """
    Parameters
    ----------
    history:
        [48,64,H,W]

    Returns
    -------
    prediction:
        [12,64,H,W]
    """

    model.eval()

    if history.shape[0] != 48:

        raise ValueError(
            "history must contain 48 months"
        )

    X = (
        history
        .unsqueeze(0)
        .to(device)
    )

    with torch.autocast(
        device_type=device.type,
        dtype=amp_dtype,
        enabled=use_amp,
    ):

        prediction = model(X)

    return (
        prediction[0]
        .float()
        .cpu()
    )


# ============================================================
# 28. COMPLETE 60-MONTH FORECAST
# ============================================================

@torch.no_grad()
def forecast_60_months(
    model,
    observed_data,
    device,
    input_length=48,
    chunk_horizon=12,
    total_horizon=60,
):
    """
    observed_data:
        [108,64,H,W]

    Returns:
        [60,64,H,W]
    """

    if (
        total_horizon
        %
        chunk_horizon
        !=
        0
    ):

        raise ValueError(
            "total_horizon must be divisible "
            "by chunk_horizon"
        )

    # --------------------------------------------------------
    # Begin with months 61-108.
    #
    # Shape:
    #
    # [48,64,H,W]
    # --------------------------------------------------------

    history = (
        observed_data[
            -input_length:
        ]
        .clone()
    )

    forecasts = []

    number_chunks = (
        total_horizon
        //
        chunk_horizon
    )

    for chunk in range(
        number_chunks
    ):

        start_month = (
            109
            +
            chunk
            *
            chunk_horizon
        )

        end_month = (
            start_month
            +
            chunk_horizon
            -
            1
        )

        print(
            f"Forecasting months "
            f"{start_month}-{end_month}"
        )

        future = (
            forecast_next_12_months(

                model=model,

                history=history,

                device=device,
            )
        )

        # future:
        #
        # [12,64,H,W]

        forecasts.append(
            future
        )

        # ----------------------------------------------------
        # Rolling 48-month context.
        #
        # Remove oldest 12.
        # Add newly predicted 12.
        # ----------------------------------------------------

        history = torch.cat(
            [
                history[
                    chunk_horizon:
                ],
                future,
            ],
            dim=0,
        )

    # --------------------------------------------------------
    # Five chunks:
    #
    # [12,...]
    # +
    # [12,...]
    # +
    # ...
    #
    # ->
    #
    # [60,64,H,W]
    # --------------------------------------------------------

    forecasts = torch.cat(
        forecasts,
        dim=0,
    )

    return forecasts


# ============================================================
# 29. FINAL PRODUCTION DATASET
# ============================================================
#
# After architecture/hyperparameter selection,
# retrain on all 49 known temporal windows.
# ============================================================

def create_final_training_loader(
    data,
    cfg,
):

    dataset = (
        FullGridTemporalDataset(

            data=data,

            time_starts=all_starts,

            input_length=(
                cfg.input_length
            ),

            horizon=(
                cfg.chunk_horizon
            ),
        )
    )

    return DataLoader(

        dataset,

        batch_size=(
            cfg.batch_size
        ),

        shuffle=True,

        num_workers=0,

        pin_memory=(
            device.type == "cuda"
        ),
    )


# ============================================================
# 30. FINAL MODEL TRAINING WITHOUT VALIDATION
# ============================================================

def train_final_model(
    model,
    loader,
    cfg,
    epochs,
):

    criterion = EmbeddingForecastLoss(
        mse_weight=(
            cfg.mse_weight
        ),
        cosine_weight=(
            cfg.cosine_weight
        ),
    )

    optimizer = torch.optim.AdamW(

        model.parameters(),

        lr=(
            cfg.learning_rate
        ),

        weight_decay=(
            cfg.weight_decay
        ),
    )

    scheduler = (
        torch.optim.lr_scheduler
        .CosineAnnealingLR(

            optimizer,

            T_max=epochs,

            eta_min=1e-6,
        )
    )

    scaler_local = torch.amp.GradScaler(
        "cuda",
        enabled=(
            use_amp
            and
            amp_dtype
            ==
            torch.float16
        ),
    )

    for epoch in range(
        1,
        epochs + 1,
    ):

        metrics = train_one_epoch(

            model=model,

            loader=loader,

            optimizer=optimizer,

            criterion=criterion,

            scaler=scaler_local,

            device=device,

            gradient_clip=(
                cfg.gradient_clip
            ),
        )

        scheduler.step()

        print(
            f"Final training "
            f"{epoch:03d}/{epochs:03d} | "
            f"loss="
            f"{metrics['loss']:.6f} | "
            f"cos="
            f"{metrics['cosine_similarity']:.4f}"
        )

    return model


# ============================================================
# 31. EXAMPLE COMPLETE WORKFLOW
# ============================================================
#
# Uncomment and adapt after loading your actual data.
# ============================================================

"""
# ------------------------------------------------------------
# LOAD DATA
# ------------------------------------------------------------

data = torch.load(
    "embeddings_grid.pt",
    map_location="cpu",
).float()

assert data.shape[0] == 108
assert data.shape[1] == 64


# ------------------------------------------------------------
# NORMALIZE EMBEDDINGS
# ------------------------------------------------------------

if cfg.normalize_embeddings:

    data = normalize_embeddings(
        data
    )


# ------------------------------------------------------------
# CREATE DEVELOPMENT LOADERS
# ------------------------------------------------------------

(
    train_loader,
    val_loader,
    test_loader,
) = make_loaders(
    data,
    cfg,
)


# ------------------------------------------------------------
# CREATE DEVELOPMENT MODEL
# ------------------------------------------------------------

model = create_model(
    cfg
).to(device)


# ------------------------------------------------------------
# TRAIN / SELECT MODEL
# ------------------------------------------------------------

best_epoch, best_val_loss = fit_model(

    model=model,

    train_loader=train_loader,

    val_loader=val_loader,

    cfg=cfg,

    checkpoint_path="best_model.pt",
)

print(
    "Best epoch:",
    best_epoch
)

print(
    "Best validation loss:",
    best_val_loss
)


# ------------------------------------------------------------
# LOAD BEST DEVELOPMENT MODEL
# ------------------------------------------------------------

checkpoint = torch.load(
    "best_model.pt",
    map_location=device,
)

model.load_state_dict(
    checkpoint[
        "model_state_dict"
    ]
)


# ------------------------------------------------------------
# TEST ON MONTHS 97-108
# ------------------------------------------------------------

test_metrics = evaluate(

    model=model,

    loader=test_loader,

    criterion=EmbeddingForecastLoss(
        mse_weight=cfg.mse_weight,
        cosine_weight=cfg.cosine_weight,
    ),

    device=device,
)

print(
    "Test loss:",
    test_metrics["loss"]
)

print(
    "Test cosine similarity:",
    test_metrics[
        "cosine_similarity"
    ]
)


# ------------------------------------------------------------
# PERFORMANCE AT EACH FORECAST HORIZON
# ------------------------------------------------------------

horizon_scores = cosine_by_horizon(

    model=model,

    loader=test_loader,

    device=device,
)

for h, score in enumerate(
    horizon_scores,
    start=1,
):

    print(
        f"Month +{h:02d}: "
        f"cosine similarity="
        f"{score.item():.4f}"
    )


# ------------------------------------------------------------
# FINAL PRODUCTION MODEL
#
# Refit using ALL 49 historical windows.
# ------------------------------------------------------------

final_loader = (
    create_final_training_loader(
        data,
        cfg,
    )
)

final_model = create_model(
    cfg
).to(device)


# A reasonable first choice is to reuse
# the best development epoch count.

final_model = train_final_model(

    model=final_model,

    loader=final_loader,

    cfg=cfg,

    epochs=best_epoch,
)


# ------------------------------------------------------------
# FORECAST MONTHS 109-168
# ------------------------------------------------------------

future_grid = forecast_60_months(

    model=final_model,

    observed_data=data,

    device=device,

    input_length=48,

    chunk_horizon=12,

    total_horizon=60,
)

print(
    "Future grid:",
    future_grid.shape
)

# Expected:
#
# [60,64,H,W]


# ------------------------------------------------------------
# CONVERT TO N GIS POINTS
# ------------------------------------------------------------

future_points = grid_to_points(
    future_grid
)

print(
    "Future points:",
    future_points.shape
)

# Expected:
#
# [N,60,64]
#
# where:
#
# N = H * W


# ------------------------------------------------------------
# SAVE
# ------------------------------------------------------------

torch.save(
    future_grid,
    "forecast_grid_60_months.pt"
)

torch.save(
    future_points,
    "forecast_points_60_months.pt"
)

np.save(
    "forecast_points_60_months.npy",
    future_points.numpy()
)
"""